# 第 1 次实验（Lab）——网站摘要浏览器

### 请先读完本节。内容偏长，但对上手很重要。

### 也请阅读 [README.md](../README.md)！更新视频说明见 README，以及[课程资源页顶部紫色区域](https://edwarddonner.com/2024/11/13/llm-engineering-resources/)

## 练习目标：你的第一个前沿 LLM 项目

课程结束时，你会搭出由 7 个智能体协作解决业务问题的 Agentic AI 方案。那是后话——我们先从更小的东西开始……

目标：写一种新型「网页浏览器」。给它一个 URL，它返回摘要——互联网版 Reader's Digest！

开始前，请先完成 README 里的环境搭建。

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions API | `openai.chat.completions.create(...)` |
| `messages`（system / user） | system 定语气与任务，user 塞网页正文 |
| 网页抓取 → 摘要 | `fetch_website_contents` + 提示词拼装 |
| 前沿小模型 | 如 `gpt-4.1-mini` / `gpt-5-nano` 等 |

## 怎么跑

1. 从上到下依次运行每个单元格（Shift+Enter）
2. `.env` 里准备好 `OPENAI_API_KEY`
3. 改 URL / 提示词，对比不同网站的摘要风格

### 如果你是 Notebook（也称 Lab / Jupyter）新手

欢迎来到数据科学实验环境！点击下方带代码的「单元格」，按 Shift+Return 执行。务必从顶部开始按顺序跑完每一格。

指南都在 [Guides 文件夹](../guides/01_intro.ipynb)。

## 我在这里帮忙

有任何问题都可以联系我。  
平台私信、邮件 ed@edwarddonner.com，或 LinkedIn：https://www.linkedin.com/in/eddonner/（欢迎连接！）  
我也在试 X：[@edwarddonner](https://x.com/edwarddonner)。

## 更多排查

请看 setup 文件夹里的 [troubleshooting](../setup/troubleshooting.ipynb)，文末有诊断脚本。

## 如果这些你已经很熟！

可以快速过前几课——后面会越来越深，最终会微调自己的 LLM 去和 OpenAI 较劲！

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">请阅读——重要说明</h2>
            <span style="color:#900;">我和你协作的方式可能和其他课程不同：我不太会边打字边让你看。我更倾向运行像这样的 Jupyter Lab，帮你建立直觉。建议你在<strong>看完讲座之后</strong>自己仔细跑一遍。多加 print 弄懂发生了什么，再做自己的变体。若有 Github，用它展示变体——既是练习，也能向未来客户或雇主证明能力……</span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">这份代码是活资源——留意我的邮件</h2>
            <span style="color:#f71;">我会定期推送更新：有人提问时会加例子或改进注释。因此下面的代码不会和视频逐字相同。视频里的内容都在；我也加了更好的解释和新模型（如 DeepSeek）。把它当成一本可交互的书。<br/><br/>
                我也会通过 Udemy 左侧「Announcements」发课程相关更新；可在 Notification Settings 里选择接收邮件。我会尽量尊重收件箱、每次都带来价值！
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">这些练习的商业价值</h2>
            <span style="color:#181;">这些笔记本既要能教，也希望好玩——比如让 LLM 讲笑话、互相抬杠。但根本目标是教会你能用在业务里的技能。路上我会点出商业含义；你一边积累模型与技法经验，一边想想今天就能在工作中落地的用法。想深入讨论或头脑风暴，欢迎联系我。</span>
        </td>
    </tr>
</table>


### 如有需要，安装 Cursor 扩展

1. 从 View 菜单选择 Extensions
2. 搜索 Python
3. 点击由 "ms-python" 提供的 "Python"，若尚未安装则选择 Install
4. 搜索 Jupyter
5. 点击由 "ms-toolsai" 提供的 "Jupyter"，若尚未安装则选择 Install


### 接下来选择 Kernel

点击右上角的 "Select Kernel"

选择 "Python Environments..."

然后选择看起来像 `.venv (Python 3.12.x) .venv/bin/python` 的那个——它应被标记为 "Recommended"，旁边有一颗大星。

有问题？请前往 troubleshooting。

### 注意：每个 notebook 都需要设置 Kernel……


In [ ]:
# ========== 导入：后面网页摘要流水线要用的工具 ==========

# 导入标准库 os：读环境变量（例如 OPENAI_API_KEY）
import os
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从本地 scraper 模块导入抓取函数：给 URL → 返回网页正文文本
from scraper import fetch_website_contents
# 从 IPython.display 导入展示工具：在笔记本里漂亮地显示 Markdown
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：调用云端 Chat Completions API
from openai import OpenAI

# 若本格报错，请去同目录 troubleshooting notebook 排查（提示保留中文教学旁注）


# 连接到 OpenAI（或 Ollama）

下一个单元格会加载你 `.env` 文件中的环境变量并连接到 OpenAI。  

如果你想改用免费的 Ollama，请查看 README 中的「Free Alternative to Paid APIs」部分；如果不确定怎么做，solutions 文件夹里有完整方案（day1_with_ollama.ipynb）。

## 遇到问题时的排查：

如果出现 "Name Error"——你是否已从顶部依次运行了所有单元格？请前往 Python Foundations 指南，那里有找到并修复所有 Name Error 的可靠方法。

如果还不行，请前往 [troubleshooting](../setup/troubleshooting.ipynb) notebook，按步骤识别根本原因并修复！

或者联系我！私信我或发邮件到 ed@edwarddonner.com，我们一起把它搞定。

担心 API 费用？请看 README 中的说明——费用应该很低，而且你可以随时控制。你也可以使用 Ollama 作为免费替代，我们会在 Day 2 讨论。


In [ ]:
# ========== 环境：加载 .env 并做 API Key 健康检查 ==========

# 加载 .env；override=True 表示已有同名环境变量也会被文件覆盖
load_dotenv(override=True)
# 从环境变量取出 OpenAI 密钥（名字必须是 OPENAI_API_KEY）
api_key = os.getenv('OPENAI_API_KEY')

# ========== 检查：常见配置错误提前喊出来 ==========

# 完全没读到密钥
if not api_key:
    # 报错文案保持英文：依赖程序判断/与排查文档对照的字符串不翻译
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
# 项目密钥通常以 sk-proj- 开头；不对就可能拿错了 key
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
# strip 后和原串不同 → 首尾有空格/制表符
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    # 基本形态看起来正常（真正能否调用还要看额度/网络）
    print("API key found and looks good so far!")


# 让我们先快速调用一次前沿模型，作为预览！


In [ ]:
# ========== 预览：构造一条最简单的 user 消息 ==========

# 发给模型的内容保持英文（可运行 / 影响回答的字符串不翻译）
message = "Hello, GPT! This is my first ever message to you! Hi!"

# Chat Completions 期望 messages 列表：这里只有一条 user
messages = [{"role": "user", "content": message}]

# 在笔记本里直接看一眼结构（最后一行表达式会显示）
messages


In [ ]:
# ========== 第一次真正调用 OpenAI ==========

# 创建客户端：默认从环境变量读 OPENAI_API_KEY
openai = OpenAI()

# chat.completions.create：指定模型 + messages，拿到一次完整回答
response = openai.chat.completions.create(model="gpt-5-nano", messages=messages)
# 助手正文在 choices[0].message.content
response.choices[0].message.content


## 好，开始我们的第一个项目


In [ ]:
# ========== 试抓取工具：给 URL，拿回网页文本 ==========

# fetch_website_contents：内部请求页面并抽出可读正文
ed = fetch_website_contents("https://edwarddonner.com")
# 打印看看抓到了什么（导航噪声可能仍在，后面靠 prompt 忽略）
print(ed)


## 提示词的类型

你可能已经知道了——如果还不知道，很快就会非常熟悉！

像 GPT 这样的模型被训练成以特定方式接收指令。

它们期望收到：

**系统提示词（system prompt）**，告诉它们正在执行什么任务、应使用什么语气

**用户提示词（user prompt）**——它们应回复的对话开场白


In [ ]:
# ========== 定义 system prompt：定角色、语气、输出格式 ==========

# 可稍后实验：把最后一句改成 Respond in markdown in Spanish. 等
# 提示词正文保持英文（改译会改变模型行为）
system_prompt = """
You are a short, humurous assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""


In [ ]:
# ========== 定义 user prompt 前缀：后面会拼上网页正文 ==========

# 前缀说明「这里有网站内容，请做短摘要」；真正正文在 messages_for 里拼接
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""


## 消息（Messages）

OpenAI 的 API 期望以特定结构接收消息。
许多其他 API 也使用这种结构：

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
先给你一个预览：接下来两个单元格会做一个相当简单的调用——我们还不会（暂时！）为难强大的 GPT


In [ ]:
# ========== 迷你演示：system 定性格，user 问算术 ==========

# messages：两条消息；role/content 字符串保持英文
messages = [
    {"role": "system", "content": "You are a rude assistant"},
    {"role": "user", "content": "What is 2 + 3?"}
]
# 用小模型做一次完整（非流式）调用
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
# 取出助手回复文本
response.choices[0].message.content


## 现在让我们用函数为 GPT-4.1-mini 构建有用的消息


In [ ]:
# ========== 组装 messages：正好是上面那种 system + user 结构 ==========

def messages_for(website):
    """把 system_prompt 与「前缀 + 网页正文」打成 API 所需列表。"""
    return [
        # system：全局角色与输出约束
        {"role": "system", "content": system_prompt},
        # user：前缀说明任务 + 抓取到的网站文本
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# ========== 试一下：对刚抓到的 ed 站点拼 messages ==========

# 返回的是列表；可再换别的网站正文试
messages_for(ed)


## 是时候把它们整合起来了——OpenAI 的 API 非常简单！


In [ ]:
# ========== 端到端：URL → 抓取 → Chat Completions → 摘要文本 ==========

def summarize(url):
    """给定网址：抓正文，调用模型，返回摘要字符串。"""
    # 1) 抓网页正文
    website = fetch_website_contents(url)
    # 2) 调 OpenAI；model / messages 参数保持原样
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    # 3) 只要助手正文
    return response.choices[0].message.content


In [ ]:
# 对 edwarddonner.com 跑一遍完整摘要
summarize("https://edwarddonner.com")


In [ ]:
# ========== 展示封装：摘要字符串 → 笔记本里的 Markdown ==========

def display_summary(url):
    """调用 summarize，再用 display(Markdown(...)) 漂亮渲染。"""
    summary = summarize(url)
    display(Markdown(summary))


In [ ]:
# 试另一个站点
display_summary("https://niranjanrajith.com")


In [ ]:
# 再对课程作者站点显示一次
display_summary("https://edwarddonner.com")


# 让我们试试更多网站

注意：这只适用于可以用这种简单方式抓取的网站。

用 Javascript 渲染的网站（如 React 应用）不会显示内容。请查看 community-contributions 文件夹中的 Selenium 实现来绕过此限制。你需要了解如何安装 Selenium（可以问 ChatGPT！）

另外，受 CloudFront（及类似服务）保护的网站可能会返回 403 错误——非常感谢 Andy J 指出这一点。

但许多网站都能正常工作！


In [ ]:
# 新闻站摘要（简单 HTTP 抓取未必总能成功）
display_summary("https://cnn.com")


In [ ]:
# Anthropic 官网摘要
display_summary("https://anthropic.com")


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">商业应用</h2>
            <span style="color:#181;">在本练习中，你第一次体验了调用前沿模型（处于 AI 前沿的领先模型）的云端 API。在本课程的许多阶段我们都会使用像 OpenAI 这样的 API，此外还会构建我们自己的 LLM。

更具体地说，我们把它用在了摘要（Summarization）上——这是生成摘要的经典 Gen AI 用例。它可以应用于任何业务领域——总结新闻、总结财务表现、把简历总结进求职信——应用场景无穷无尽。想想你如何在业务中应用摘要，并尝试制作一个原型。</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">继续之前——现在自己试试</h2>
            <span style="color:#900;">使用下方单元格制作你自己的简单商业示例。目前先继续使用摘要用例。这里有一个想法：写一个程序，接收邮件内容，并建议一个合适的简短主题行。这正是可能被集成到商业邮件工具中的功能类型。</span>
        </td>
    </tr>
</table>


In [ ]:
# ========== 步骤 1：为自己的业务小例子写 prompts ==========

# system：让模型分析聊天记录里经理/员工的专业度（提示词保持英文）
system_prompt = """
You are a expert assistant that analyzes the contents of a chat,
and provide a short character analaysis of professionalism for the manager  and employee , ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""



# user：真实聊天语料作为「数据」喂给模型（全文保持原样，勿翻译）
user_prompt = """
   Manager  [8:40 AM]
GM @Emp_A.immadi
[8:40 AM]Can you help with reproducing this issue today for client
Manager  [9:59 AM]
issues/59442#issuecomment-194923181
issues/57644 - also I dont see an update here, can you make sure you update this before 10.30 am scrum
Employee_A  [10:18 AM]
updated the issue
Manager  [10:32 AM]
ok
Manager  [10:42 AM]
Hello @Emp_A.immadi
[10:42 AM]let me know when we can talk for 10 mis for the customer case
Employee_A  [11:26 AM]
Hi Manager
[11:26 AM]call?
Manager  [11:29 AM]
yes
A huddle happened  [11:31 AM]
You and Employee_A were in the huddle for 6m.Employee_A  [11:37 AM]
racker/issues/60349
Manager  [11:38 AM]
tracker/issues/60332
tracker/issues/59442
Manager  [11:46 AM]
Hello @Emp_A.immadi
[11:46 AM]can you check this - tracker/issues/60319
for 5.3.1 patch 6 - need to be done todayManager  [12:35 PM]
Hello @Emp_A.immadi
[12:36 PM]hope you saw this client meeting invite for today 5.30 p,m
Employee_A  [12:41 PM]
Hi Manager
[12:41 PM]hope you saw this client meeting invite for today 5.30 p,myes, will be joining
[12:42 PM]for 5.3.1 patch 6 - need to be done todaychecking
Manager  [12:43 PM]
thanks Emp_A
Manager  [8:35 AM]
Hello Emp_A, how did the client meeting go?
Employee_A  [10:16 AM]
can you check this - git_url:/tracker/issues/60319git_url:/aios-payload-logging-service/pull/1373
giturl:aios-payload-logging-service-api/pull/1202
raised pr's for the issue[10:19 AM]Hello Emp_A, how did the client meeting go?I was able to find the cause of the error on thursday in client call, Have asked harivansh to join the call on friday. And we have further minimised the reasons why this could happen and have shared the sequence of steps to w'll follow to resolve the isse.
[10:19 AM]Calling out action items here:

Get DevOps involved to help investigate why call is not coming through to PL-API pods.
Emp_A to try and reproduce the scenario in-house with focus on replicating same volume of data.
OpenScale team to look at count query and identify optimization and improvements to improve query response time.
Brij to check if creating another subscription is an option as we confirmed in call that GET /records is failing only for this subscription and not for other.
From a private conversation | May 8thEmployee_A  [6:10 PM]
git_url:/aios-payload-logging-service/pull/1373
giturl:aios-payload-logging-service-api/pull/1202
could we merge these?Manager  [6:12 PM]
yes , approved
Employee_A  [6:12 PM]
i don't have permission to merge
Manager  [6:14 PM]
Done
[6:14 PM]What are your priority tasks now Emp_A
[6:15 PM]What happened with PG-blocker and CSV splitting
[6:15 PM]can you share methose git issues
Employee_A  [6:53 PM]
git_url:/tracker/issues/53851
last time i validated through ui and sdk is pending, did not check after that[6:54 PM]i have tried sdk but the changes did not work, could be because of the sdk version i was using
Manager  [10:35 AM]
@Emp_A.immadi are you on sayaka's call?
Employee_A  [10:36 AM]
joining
Manager  [10:37 AM]
Can you update this in git_url:/tracker/issues/59442
image.png Employee_A  [10:38 AM]
this is different task
Manager  [11:00 AM]
which issue is this then?
[11:00 AM]For 59442 after the customer call - can you consolidate what are the actions in progress
Employee_A  [11:02 AM]
sure, give me few mins
Employee_A  [11:10 AM]
git_url:/tracker/issues/59442#issuecomment-197821262
[11:10 AM]this call is at 10 30 am
[11:13 AM]after multiple attempts i was not able to run into this 403 error
[11:14 AM]this call is to understand and see if we could reproduce the issue
Manager  [11:16 AM]
ok, so you could reproduce issue at your end
[11:16 AM]couldn't
Employee_A  [11:18 AM]
yes, we couldn't. same as the other issue where @suraj.kumar24 @kshitij.g1 worked
Manager  [11:20 AM]
Can you update that in the issue
Employee_A  [11:21 AM]
but this is a different error user is seeing.
Manager  [11:21 AM]
can we connect?
Employee_A  [11:21 AM]
sure
A huddle happened  [11:22 AM]
You and Employee_A were in the huddle for 2m.Manager  [9:51 AM]
Hello @Emp_A.immadi
[9:51 AM]image.png [9:51 AM]what are these details
[9:52 AM]Is this the erro screenshot?
[9:52 AM]why this Sorry and all in bold?  Cookies info - is that really relevant ? I understand that yu are able reproduce the upload issue now. Earlier you were not hitting with it and erro was coming Quality evaluation only. (edited) 
Employee_A  [10:01 AM]
Hi Manager
[10:04 AM]this is the response from risk_evaluations api, It tells users ip is blocked by cloud flare to upload the records as it found some malicious characters in the payload.
[10:05 AM]yes, after the customer call we got more information on how to reproduce the issue.
Manager  [10:05 AM]
ok
Manager  [4:08 PM]
Emp_A - git_url:/tracker/issues/58790 - Tensorflow model support - Any discussion happened for this with Harivansh?
Employee_A  [7:21 PM]
@Managerrajith Please review
git_url:/aios-payload-logging-service-api/pull/1208#issuecomment-198579325
[7:22 PM]Emp_A - git_url:/tracker/issues/58790 - Tensorflow model support - Any discussion happened for this with Harivansh?i will be working on this
[7:22 PM]did not actively spend time as i was checking other customer issues
[7:24 PM]git_url:/tracker/issues/60349
git_url:/tracker/issues/60574
git_url:/tracker/issues/59442 (edited) 
Manager  [9:48 PM]
approved
Employee_A  [9:49 PM]
please merge as well
Manager  [9:47 AM]
Done
Employee_A  [10:56 AM]
git_url:/aios-payload-logging-service-api/pull/1211
[10:56 AM]please merge this
Manager  [10:59 AM]
Done
Manager  [4:52 PM]
Hello @Emp_A.immadi
[4:53 PM]are you around?
Employee_A  [4:59 PM]
Hi Manager
[4:59 PM]yes
Manager  [5:14 PM]
tracker/issues/60574
A huddle happened  [5:16 PM]
You and Employee_A were in the huddle for 3m.Manager  [5:16 PM]
calling
Employee_A  [5:37 PM]
issues/60574#issuecomment-198991654
Manager  [7:04 PM]
Hello @Emp_A.immadi
[7:04 PM]are you still around?
Employee_A  [11:03 AM]
Hi Manager
Employee_A  [11:08 AM]
Have msged @Harivansh regarding pr's, waiting for reply.
[11:09 AM]Hey Folks,
Good Morning..

Please let me know what the next steps are here... While including the fix for missing index in 5.4 is important, I want to understand what are we doing for the open sev 1 issue.. Are they expecting a hot fix? Are they OK to close the issue and move to Batch?
From a private conversation | May 18th[11:09 AM]arun has shared this in morning
[11:09 AM]In addition to the missing index, we also need to have a proper query.. Both of them have to be fixed.. Please ensure that..
From a private conversation | May 18thManager  [11:14 AM]
Thanks
[11:15 AM]Hope all Arun's concerns taken care
Manager  [1:53 PM]
@Emp_A.immadi please update on this once you are back https://xx-comp-analytics.slack.com/archives/C0B3ZJJGKUM/p1779265360992789
@Emp_A.immadi - is this behaviour for missing create table api part of create datasets, the same in SaaS?
Direct Message | May 20th | View conversationEmployee_A  [2:42 PM]
Hi Manager
[2:43 PM]will update after the meeting with harivansh
Manager  [2:49 PM]
ok
Employee_A  [7:19 PM]
issues/61088
https://dptform.cloud.xx-comp.com/del/wos-manage-payload-data.html?context=wx
Hi Manager, incase we need to update the documentation. How can we do that?
Manager  [8:17 PM]
we need to raise a documentation issue and assign it to @carol.wiest
[8:18 PM]issues/new/choose
[8:18 PM]you willl see a doc issue template here
Employee_A  [1:06 PM]
Hi Manager, could not join the scrum today. Not Feeling well
Manager  [1:23 PM]
ok, Emp_A. Take rest and get well
Manager  [10:04 AM]
Hi @Emp_A.immadi
[10:04 AM]can you take car eof this today - /issues/60992
[10:05 AM]How is your sinus diagnosis going on?
Employee_A  [10:08 AM]
Gm Manager
[10:11 AM]How is your sinus diagnosis going on?Was using medication from last couple of weeks and doctor suggested to lose some weight. He is suggesting to delay surgery as i have recently undergone one.
Manager  [10:14 AM]
ok
[10:16 AM]Plan well on your health.. Gym will help  rt?
Manager  [10:23 AM]
hissues/41744 - can this be included in 5.4 patch 2 - DCUT June 15th
Employee_A  [10:25 AM]
Will have to check on this again, will let you know if i could spend sometime and can include in 5.4 patch 2.
Manager  [10:26 AM]
can you confirm on this by tomorrow
Employee_A  [10:26 AM]
sure
Manager  [6:26 PM]
Hello @Emp_A.immadi
[6:26 PM]there is a prod PL issue - which need attention, Nith awas handling it
[6:26 PM]But now she completely occupied with her gov 2.0 work
[6:27 PM]wanted to check if you have soem time to check that
[6:27 PM]on Friday
[6:27 PM]issues/52983#issuecomment-202564815
Employee_A  [6:51 PM]
Hi Manager
[6:52 PM]let me take a look
"""

# ========== 步骤 2：打成 messages 列表（system + user）==========

messages = [
            {"role":"system","content":system_prompt},
            {"role":"user","content" :user_prompt}
            ] # fill this in

# ========== 步骤 3：调用 OpenAI Chat Completions ==========

# model id 保持原样；messages 即上一步组装的列表
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)

# ========== 步骤 4：打印助手正文 ==========

print(response.choices[0].message.content)
# 也可改成 display(Markdown(...)) 更漂亮地渲染


## 额外练习：喜欢网页抓取的同学可以继续挑战

你可能发现：若试 `display_summary("https://openai.com")`——会失败！因为 OpenAI 官网大量使用 Javascript。绕过办法很多，例如 **Selenium**：在后台跑浏览器、渲染页面后再查询 DOM。如果你熟悉 Selenium、Playwright 等，可以改进 Website / 抓取类去用它们。`community-contributions` 文件夹里有学员贡献的 Selenium 示例（感谢！）。


# 分享你的代码

我很希望你之后分享代码，这样我就能分享给其他人！你会注意到一些学员已经做了改动（包括 Selenium 实现），可以在 community-contributions 文件夹中找到。如果想把你的改动加入该文件夹，请提交包含新版本的 Pull Request，我会合并你的更改。

如果你不是 git 专家（我也不是！），我在 guides 文件夹的指南 3 中给出了完整说明，并粘贴在这里：

以下是创建 PR 的整体步骤和关键说明：  
https://edwarddonner.com/pr  

提交前请检查：  
1. 你的 PR 只包含 community-contributions 中的更改（除非我们另有讨论）  
2. 所有 notebook 输出已清空  
3. 总计少于 2,000 行代码，文件也不要太多  
4. 不要包含不必要的测试文件、过于冗长的 README 或 .env.example，也不要 emoji 或其他 LLM 产物！

非常感谢！

详细步骤在此： 

https://chatgpt.com/share/6873c22b-2a1c-8012-bc9a-debdcf7c835b
